In [ ]:
# ==============================================================================
# PHẦN 1: CÀI ĐẶT MÔI TRƯỜNG & KẾT NỐI DRIVE
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

# Cài đặt phiên bản Ultralytics mới nhất và nest_asyncio
!pip install -U ultralytics nest_asyncio

import ultralytics
from ultralytics import YOLO, hub
import os
import nest_asyncio

# --- ĐIỂM CHỐT HẠ LỖI Ở ĐÂY ---
nest_asyncio.apply()
# ------------------------------

# Kiểm tra sức mạnh GPU trên Colab Pro
ultralytics.checks()

# ==============================================================================
# PHẦN 2: KẾT NỐI ULTRALYTICS PLATFORM BẰNG URI CHUẨN
# ==============================================================================
# 1. Ép hệ thống nhận diện API Key như một Biến Môi Trường
os.environ["ULTRALYTICS_API_KEY"] = "ul_58eaafc88c4778e76e600ce593c552699002a468"

# 2. Đăng nhập dự phòng cho các tiến trình khác của HUB
hub.login(os.environ["ULTRALYTICS_API_KEY"])

# 3. Khai báo URI
DATASET_URI = "ul://violet/datasets/work3yolov8"

# ==============================================================================
# PHẦN 3: HUẤN LUYỆN YOLO26n (Tối ưu cho Edge AI)
# ==============================================================================
# Khởi tạo mô hình YOLO26n (Nano) với kiến trúc NMS-Free
model = YOLO("yolo26n.pt")

# Bắt đầu Train với GPU L4
results = model.train(
    data=DATASET_URI,
    epochs=100,
    imgsz=640,
    batch=32,
    workers=8,
    cache=True,
    optimizer='MuSGD',
    device=0,
    project='/content/drive/MyDrive/Traffic_AI_Project',
    name='YOLO26n_Traffic_Native',
    exist_ok=True,
    patience=50,
    save=True,
    augment=True,
    plots=True
)

# ==============================================================================
# PHẦN 4: XÁC THỰC (VALIDATION) & XUẤT MÔ HÌNH SANG ONNX
# ==============================================================================
metrics = model.val()
print(f"Độ chính xác mAP 50-95: {metrics.box.map}")

print("--- Đang xuất mô hình sang ONNX (Định dạng trung gian) ---")
success = model.export(
    format='onnx',
    simplify=True,
    dynamic=False
)

print(f"Hoàn tất! Hãy tải file best.pt trong thư mục {results.save_dir}/weights về Jetson Nano.")
print("Lưu ý: Mở Terminal trên Jetson Nano và chạy lệnh sau để lấy file TensorRT:")
print("yolo export model=best.pt format=engine half=True workspace=2")

Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
Setup complete ✅ (12 CPUs, 53.0 GB RAM, 43.0/235.7 GB disk)
requirements: Ultralytics requirement ['hub-sdk>=0.0.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 6 packages in 223ms
Prepared 1 package in 19ms
Installed 1 package in 6ms
 + hub-sdk==0.0.24

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



2026-04-18 17:33:55,622 - hub_sdk.helpers.logger - WARNING - Ultralytics HUB-SDK: Invalid API key ⚠️


Ultralytics HUB: Get API key from https://hub.ultralytics.com/settings?tab=api+keys and then run 'yolo login API_KEY'
Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=ul://violet/datasets/work3yolov8, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_s

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.91...
ONNX: export success ✅ 6.2s, saved as '/content/drive/MyDrive/Traffic_AI_Project/YOLO26n_Traffic_Native/weights/best.onnx' (9.4 MB)

Export complete (6.5s)
Results saved to /content/drive/MyDrive/Traffic_AI_Project/YOLO26n_Traffic_Native/weights
Predict:         yolo predict task=detect model=/content/drive/MyDrive/Traffic_AI_Project/YOLO26n_Traffic_Native/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/Traffic_AI_Project/YOLO26n_Traffic_Native/weights/best.onnx imgsz=640 data=/content/datasets/work3yolov8/data.yaml  
Visualize:       https://netron.app
Hoàn tất! Hãy tải file best.pt trong thư mục /content/drive/MyDrive/Traffic_AI_Project/YOLO26n_Traffic_Native/weights về Jetson Nano.
Lưu ý: Mở Terminal trên Jetson Nano và chạy lệnh sau để lấy file TensorRT:
yolo export model=best.pt format=engine half=True workspace=2
